### OOM Analysis
the actual results are too large so they are under .gitignore \
make sure you run this command before running the notebook to unzip the results: \
``--tar -xvzf results_intermediate.tar.gz``

Convert the results into pandas dataframes for analysis

In [28]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/Users/beliz/Desktop/thesis_project/SagiXGBoostTreeApproximator")
status_df = pd.read_csv(PROJECT / "farm1" / "status.txt", sep=" ", header=None, names=["run_id", "status"])

# convert to integer columns
status_df["run_id"] = status_df["run_id"].astype(int)
status_df["status"] = status_df["status"].astype(int)
status_df.head()

,run_id,status
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [29]:
import pandas as pd

rows = []

with open(PROJECT / "farm1" / "table.dat", "r") as f:
    for line in f:
        parts = line.strip().split()

        row = {}
        
        # first parts (fixed structure)
        row["id"] = parts[0]
        row["command"] = parts[1]
        row["script"] = parts[2]

        # parse flags (key-value pairs)
        i = 3
        while i < len(parts):
            if parts[i].startswith("--"):
                key = parts[i][2:]  # remove "--"
                value = parts[i+1] if i+1 < len(parts) else None
                row[key] = value
                i += 2
            else:
                i += 1

        rows.append(row)

command_df = pd.DataFrame(rows)
command_df["id"] = command_df["id"].astype(int)
command_df["fold"] = command_df["fold"].astype(int)
command_df["trial-id"] = command_df["trial-id"].astype(int)

command_df.head()

,id,command,script,dataset,fold,trial-id,home-dir
0,1,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,0,1,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
1,2,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,1,1,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
2,3,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,2,1,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
3,4,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,3,1,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
4,5,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,4,1,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...


We are interested in the OOM cases: 0 = successful. 127 = timeout (dw about these), 137 = OOM. 

In [30]:
oom_status_df = status_df[status_df["status"] == 137]
print(oom_status_df.shape)
oom_status_df.head()

(740, 2)


,run_id,status
16395,16396,137
16396,16397,137
16397,16398,137
16398,16399,137
16399,16400,137


In [31]:
# now we want to filter the command_df for the OOM cases
oom_command_df = command_df[command_df["id"].isin(oom_status_df["run_id"])]
print(oom_command_df.shape) # sanity check
oom_command_df.head()


(740, 7)


,id,command,script,dataset,fold,trial-id,home-dir
16395,16396,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,0,80,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
16396,16397,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,1,80,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
16397,16398,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,2,80,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
16398,16399,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,3,80,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...
16399,16400,python,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...,room,4,80,/home/upadhyan/scratch/SagiXGBoostTreeApproxim...


In [32]:
# Now we want to see count how OOM cases are distributed for trial-ids and datasets
# First, let's get the unique trial-ids and datasets
trial_ids = oom_command_df["trial-id"].unique()
datasets = oom_command_df["dataset"].unique()

# Now we want to count how many OOM cases there are for each trial-id and dataset
trial_id_counts = oom_command_df["trial-id"].value_counts()
dataset_counts = oom_command_df["dataset"].value_counts()


In [33]:
# See results for Trial IDs
print("Num of unique trial-ids: ", len(trial_ids))
print("Trial IDs with OOM cases: ", trial_ids)
print("\nTrial ID counts: ", trial_id_counts)

Num of unique trial-ids:  145
Trial IDs with OOM cases:  [ 80   1   2   3   4   5  10  11  12  13  14  16  17  18  19  20  21  22
  24  25  29  30  31  32  33  36  38  39  40  41  42  43  44  45  46  47
  48  49  50  53  55  58  60  61  62  64  67  68  69  71  73  75  77  78
  79  81  83  84  85  86  87  88  89  90  91  93  94  95  98  99 100 102
 103 104 107 109 110 111 112 114 116 119 120 121 122 124 125 126 127 128
 129 130 131 132 133 134 135 136 137 138 139 140 141 142 144 145 147 148
 149 150 151 152 153 154 155 159 160 163 166 167 168 169 171 173 174 175
 178 179 180 181 182 183 185 186 187 189 190 193 194 195 196 197 198 199
 200]

Trial ID counts:  trial-id
80     10
1      10
4      10
10     10
17     10
       ..
127     4
171     3
84      2
155     2
186     1
Name: count, Length: 145, dtype: int64


In [34]:
# See results for Datasets
print("\nNum of unique datasets: ", len(datasets))
print("\nDataset counts: ", dataset_counts)


Num of unique datasets:  3

Dataset counts:  dataset
avila    706
bean      29
room       5
Name: count, dtype: int64


We can see that the issue is concentrated in the avila dataset. Now we want to explore how much of the runs done with avila experienced this issue.

In [35]:
(command_df[command_df["dataset"] == "avila"]).shape

(2000, 7)

**Class counts (supervisor hypothesis)** — OOM might correlate with **many classes** and/or **strong imbalance** (tiny minority classes), which can affect boosting / pruning / conjunction growth. Below we load each farm dataset the same way as `tests/xgb_run_new.py` (`DataFactory_clf`) and summarize label frequencies. *First run may download/cache OpenML data into `PROJECT/data/`.*

In [36]:
import importlib.util
import numpy as np

# Load tests/data_utils.py by path (notebook cwd is often farm1/, not tests/).
_data_utils_path = PROJECT / "tests" / "data_utils.py"
_spec = importlib.util.spec_from_file_location("tests_data_utils", _data_utils_path)
_data_utils = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_data_utils)
DataFactory_clf = _data_utils.DataFactory_clf

CACHE_DIR = str(PROJECT / "data")


def label_value_counts(dataset: str) -> pd.Series:
    """Same preprocessing as farm jobs: encoded integer labels after LabelEncoder."""
    factory = DataFactory_clf(dataset=dataset, cache_dir=CACHE_DIR)
    return (
        pd.Series(factory.y, name="label")
        .value_counts()
        .sort_index()
        .rename_axis("class_index")
    ) # How many points fall in each encoded class, sorted by class index. class frequencies for the labels actually used in training.


def imbalance_metrics(vc: pd.Series, n_samples: int) -> dict:
    counts = vc.values.astype(float)
    k = len(counts)
    min_c = counts.min()
    max_c = counts.max()
    return {
        "n_classes": k,
        "min_class_count": int(min_c),
        "max_class_count": int(max_c), # smallest and largest class sizes.
        "imbalance_ratio (max/min)": float(max_c / min_c) if min_c > 0 else np.inf, # how much the largest class dominates the smallest (high ⇒ very imbalanced).
        "minority_frac": float(min_c / n_samples),
        "majority_frac": float(max_c / n_samples), # smallest (largest) class size divided by n_samples.
    } # how many classes and how skewed are the sizes of these classes


farm_datasets = sorted(command_df["dataset"].unique())
rows = []
vc_by_dataset: dict[str, pd.Series] = {}

for ds in farm_datasets:
    try:
        vc = label_value_counts(ds)
        n = int(vc.sum())
        m = imbalance_metrics(vc, n)
        rows.append({"dataset": ds, "n_samples": n, **m})
        vc_by_dataset[ds] = vc
    except Exception as e:
        print(f"{ds!r}: skipped ({e})")

class_summary_df = pd.DataFrame(rows)

# Join with OOM frequency (same jobs table as above)
n_per_ds = command_df.groupby("dataset").size().rename("n_jobs")
oom_per_ds = oom_command_df.groupby("dataset").size().rename("n_oom")
class_summary_df = class_summary_df.set_index("dataset").join([n_per_ds, oom_per_ds], how="left")
class_summary_df["n_oom"] = class_summary_df["n_oom"].fillna(0).astype(int)
class_summary_df["oom_rate"] = class_summary_df["n_oom"] / class_summary_df["n_jobs"]

class_summary_df = class_summary_df.reset_index().sort_values(
    ["oom_rate", "imbalance_ratio (max/min)", "n_classes"],
    ascending=[False, False, False],
)
class_summary_df

,dataset,n_samples,n_classes,min_class_count,max_class_count,imbalance_ratio (max/min),minority_frac,majority_frac,n_jobs,n_oom,oom_rate
0,avila,20867,12,10,8572,857.200000,0.000479,0.410792,2000,706,0.3530
2,bean,13611,7,522,3546,6.793103,0.038351,0.260525,2000,29,0.0145
12,room,10129,4,459,8228,17.925926,0.045315,0.812321,2000,5,0.0025
9,page,5473,5,28,4913,175.464286,0.005116,0.897680,2000,0,0.0000
15,wilt,4839,2,261,4578,17.540230,0.053937,0.946063,2000,0,0.0000
5,fault,1941,7,55,673,12.236364,0.028336,0.346728,2000,0,0.0000
6,htru,17898,2,1639,16259,9.920073,0.091574,0.908426,2000,0,0.0000
3,bidding,6321,2,675,5646,8.364444,0.106787,0.893213,2000,0,0.0000
14,skin,245057,2,50859,194198,3.818361,0.207539,0.792461,2000,0,0.0000
8,occupancy,20560,2,4750,15810,3.328421,0.231031,0.768969,2000,0,0.0000


In [37]:
# Per-class counts for datasets that actually OOM'd (compare shapes side by side)
for ds in ["avila", "bean", "room"]:
    if ds in vc_by_dataset:
        print(f"\n=== {ds} ===")
        print(vc_by_dataset[ds].to_string())
        print("total", int(vc_by_dataset[ds].sum()))


=== avila ===
class_index
0     8572
1       10
2      206
3      705
4     2190
5     3923
6      893
7     1039
8     1663
9       89
10    1044
11     533
total 20867

=== bean ===
class_index
0    1322
1     522
2    1630
3    3546
4    1928
5    2027
6    2636
total 13611

=== room ===
class_index
0    8228
1     459
2     748
3     694
total 10129


In [38]:
import sys
sys.executable

'/Users/beliz/Desktop/thesis_project/SagiXGBoostTreeApproximator/venv/bin/python'

In [40]:
avila_oom = oom_command_df[oom_command_df["dataset"] == "avila"]
# export to csv
avila_oom.to_csv("avila_oom.csv", index=False)
